# Prerequisites


Load 'Employee.csv' into DataFrame

In [0]:
df = spark.read.csv(path="/Volumes/merit_catalog/quickstart_schema/sandbox/dataset/employee.csv",header=True,inferSchema=True,sep="|",quote="'")
df.display()

### Select Columns

In [0]:
df.select("id","name").display()

# from pyspark.sql.functions import col
# df.select(col("id"),col("name").alias("FullName")).display()

### Fliter Records

In [0]:
# df.filter(col("gen")=="M").select(col("name")).display()

from pyspark.sql.functions import col, lower
df.filter(lower("company")=="cisco").select("name").display()

In [0]:
df.filter((col("gen")=="M") & (col("company")=="Infosys")).display()

### New column
The lit() function in PySpark is used to create a column containing a literal (constant)

In [0]:
from pyspark.sql.functions import lit
df.withColumn("is_employed", lit("True")).display()


In [0]:
from pyspark.sql.functions import col, when

df.withColumn(
    "experience_category",
    when(col("exp") >= 10, "Senior")
    .when(col("exp") >= 5, "Mid Level")
    .when(col("exp") >= 0, "Junior")
    .otherwise("invalid")
).select("name", "exp", "experience_category").display()

In [0]:
df.withColumn(
    "gen",
    when(col("gen")=="M","Male")
    .when(col("gen")=="F","Female")
    .when(col("gen")=="T","Transgender")
    .otherwise("invalid")
).display()

GroupBY and sort

In [0]:
df.withColumn(
    "experience_category",
    when(col("exp") >= 10, "Senior")
    .when(col("exp") >= 5, "Mid Level")
    .when(col("exp") >= 0, "Junior")
    .otherwise("invalid")
)
df.groupBy("experience_category").count().sort("count", ascending=False).display()

In [0]:
df.withColumn(
    "experience_category",
    when((col("exp") >= 10) & (col("gen")=="M"), "Senior")
    .when((col("exp") >= 9) & (col("gen")=="F"), "Senior")
    .when((col("exp") >= 5) & (col("gen")=="M"), "Mid Level")
    .when((col("exp") >= 4) & (col("gen")=="F"), "Mid Level")
    .when((col("exp") >= 0) & (col("gen")=="M"), "Junior")
    .when((col("exp") >= 0) & (col("gen")=="F"), "Junior")
    .when((col("exp") >= 10) & (col("gen")=="T"), "Senior")
    .when((col("exp") >= 5) & (col("gen")=="T"), "Mid Level")
    .when((col("exp") >= 0) & (col("gen")=="T"), "Junior")
    .otherwise("invalid")
).select("name","gen","exp","experience_category").display()

In [0]:
from pyspark.sql.functions import col, when

df.withColumn(
    "experience_category",
    when(
        (
            ((col("exp") >= 10) & (col("gen") == "M")) |
            ((col("exp") >= 9) & (col("gen") == "F"))
        ),
        "Senior"
    )
    .when(
        (
            ((col("exp") >= 5) & (col("gen") == "M")) |
            ((col("exp") >= 4) & (col("gen") == "F"))
        ),
        "Mid Level"
    )
    .when(
        (col("exp") >= 0),  # simplified
        "Junior"
    )
    .otherwise("invalid")
).display()